# PriorModel — Colab (TE+TM, dx=80 m)

U-Net eğitimi Colab GPU'da; COMMEMI / VFSA değerlendirmesi **yerelde** kalır (`scripts/evaluate_mid_scale_v8.jl`). `main.jl` MTGeophysics → GLMakie yükler ve headless Colab'da takılır.

**Bu koşu:** `train_pairs_v8_tetm_n200.h5` (n=200, 4 kanal TE+TM, 48×240) → `models/prior_v8_tetm_n200.jld2`, 50 epoch. T4 16 GB yeter.

1. `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 yeterli).
2. Kernel **Python 3** kalsın. Julia `%%bash` hücrelerinden `julia --project=/content/PriorModel …` ile çağrılır.
3. GitHub `master`'daki bu defteri açın veya `File → Upload notebook` ile yükleyin.

Yerelde ürettiğiniz HDF5 varsa Drive'a koyun:

`MyDrive/PriorModelData/PriorModel/PriorModel/data/synthetic/train_pairs_v8_tetm_n200.h5`

## `--project=.` kullanmayın

Colab çalışma dizini `/content`. Oradan `julia --project=.` boş bir `/content/Project.toml` bırakır; sonra `using HDF5` / `using LuxCUDA` "not found in current path" verir.

Her zaman klon yolunu verin:

```bash
julia --project=/content/PriorModel …
```

In [ ]:
%%bash
set -euo pipefail
if [[ -f /content/Project.toml ]]; then
  echo "Removing leftover /content/Project.toml (created by --project=.)"
  rm -f /content/Project.toml /content/Manifest.toml
fi
ls -la /content/Project.toml 2>/dev/null || echo "No /content/Project.toml — good."

## Julia 1.12.4 kur

Colab'ın yerleşik Julia runtime'ı 1.10 LTS; bu proje Julia 1.11+ ister (`Project.toml`). 1.12.4 Linux x86_64 tarball'ını `/usr/local`'a açıyoruz.

In [ ]:
%%bash
set -euo pipefail
JULIA_VERSION="1.12.4"
JULIA_VER="${JULIA_VERSION%.*}"
if julia --version 2>/dev/null | grep -q "${JULIA_VERSION}"; then
  julia --version
  exit 0
fi
echo "Installing Julia ${JULIA_VERSION}…"
URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_VER}/julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
wget -q "${URL}" -O /tmp/julia.tar.gz
tar -xzf /tmp/julia.tar.gz -C /usr/local --strip-components=1
rm /tmp/julia.tar.gz
julia --version

## Repoyu klonla

Public HTTPS clone kullanıcı adı sormasın diye `GIT_TERMINAL_PROMPT=0`. 90 saniye takılırsa `master.zip` indirilir.

In [ ]:
%%bash
set -euo pipefail
export GIT_TERMINAL_PROMPT=0
export GIT_PAGER=cat
unset GIT_ASKPASS SSH_ASKPASS

REPO_URL="https://github.com/hayrunnisayildiz/PriorModel.git"
ZIP_URL="https://github.com/hayrunnisayildiz/PriorModel/archive/refs/heads/master.zip"
DEST="/content/PriorModel"

echo "=== public clone (no login) ==="

if [[ -d "${DEST}/.git" ]]; then
  echo "Updating existing clone…"
  git -C "${DEST}" -c credential.helper= --no-pager fetch --depth 1 origin master
  git -C "${DEST}" --no-pager reset --hard FETCH_HEAD
else
  rm -rf "${DEST}"
  echo "Cloning ${REPO_URL} …"
  if timeout 90 git -c credential.helper= clone --depth 1 --single-branch --branch master --progress "${REPO_URL}" "${DEST}"; then
    echo "git clone OK"
  else
    echo "git clone stalled/failed — downloading public zip instead"
    rm -rf "${DEST}"
    wget -q --show-progress -O /tmp/PriorModel.zip "${ZIP_URL}"
    unzip -qo /tmp/PriorModel.zip -d /tmp
    mv /tmp/PriorModel-master "${DEST}"
    rm -f /tmp/PriorModel.zip
  fi
fi

test -f "${DEST}/Project.toml"
ls -l "${DEST}/Project.toml"
echo "OK"

## Mesh doğrula (yama yok)

`UNET_MESH` GitHub `master`'da zaten **nx=240, dx=80 m**. Eski 120×160 yaması artık gerekmez.

In [ ]:
from pathlib import Path

p = Path("/content/PriorModel/src/synthetic/MeshParams.jl")
text = p.read_text()
start = text.find("const UNET_MESH")
assert start >= 0, "UNET_MESH not found in MeshParams.jl"
block = text[start:start + 700]
print(block)
assert "240," in block and "80.0," in block, block
assert "160.0," not in block, "UNET_MESH still has dx=160 m"
print("OK: UNET_MESH is 240 × 80 m")

## Google Drive'ı bağla

HDF5 ve checkpoint'ler gitignore'da. Hücre şu kökleri sırayla arar:

- `/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel/`
- `/content/drive/MyDrive/PriorModel/`

Hedef dosya: `data/synthetic/train_pairs_v8_tetm_n200.h5` (v7 48×120 — bu koşuda kullanmayın).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Julia projesini instantiate et

`MTGeophysics.jl` v0.4.2 GitHub tag'inden gelir (`Project.toml` `[sources]`). GLMakie hard dependency; `src/pkg_setup.jl` Colab'da auto-precompile'ı kapatır. Burada `using MTGeophysics` yapmayın.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
echo "=== Pkg.instantiate (several minutes; no output is normal) ==="
julia --project=/content/PriorModel -e '
println("julia started; instantiating…"); flush(stdout); flush(stderr)
include("/content/PriorModel/src/pkg_setup.jl")
println("active=", Base.active_project())
flush(stdout)
'

## GPU kontrolü

Colab runtime GPU olsa bile eğitim yalnızca Julia `CUDA.functional() = true` ise GPU kullanır. `Device: CPU` veya `LuxCUDA unavailable` görürseniz Runtime tipini GPU yapıp kernel'ı restart edin.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul

echo "=== Colab runtime ==="
nvidia-smi -L || { echo "nvidia-smi missing: Runtime is not GPU. Runtime → Change runtime type → GPU, then restart."; exit 1; }
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

echo
echo "=== Julia CUDA ==="
julia --project=/content/PriorModel -e '
println("julia started"); flush(stdout)
try
    using LuxCUDA
catch err
    println("LuxCUDA failed: ", err)
    println("→ training will be CPU")
    exit(1)
end
println("CUDA.functional() = ", CUDA.functional())
if CUDA.functional()
    println("GPU = ", CUDA.name(CUDA.device()))
    println("OK: training will use CUDA GPU")
else
    println("CUDA.jl loaded but no usable GPU (CUDA.functional()=false)")
    try
        CUDA.versioninfo()
    catch e
        println(e)
    end
    exit(1)
end
'

## Drive'dan TE+TM HDF5 kopyala

Dosya yoksa sonraki hücre `xvfb-run` ile `--tetm --n 200` üretir (~30–60 dk, TE+TM forward). Drive'da varsa bu hücre `OK` basar ve build'i atlayın.

In [ ]:
from pathlib import Path
import shutil

NAME = "train_pairs_v8_tetm_n200.h5"
DST_DIR = Path("/content/PriorModel/data/synthetic")
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel"),
    Path("/content/drive/MyDrive/PriorModelData/PriorModel"),
    Path("/content/drive/MyDrive/PriorModel"),
]

DST_DIR.mkdir(parents=True, exist_ok=True)
dst = DST_DIR / NAME

print("=== copy TE+TM HDF5 from Drive ===", flush=True)
found = None
for root in CANDIDATE_ROOTS:
    src = root / "data" / "synthetic" / NAME
    print(f"  check {src}  exists={src.is_file()}", flush=True)
    if src.is_file():
        found = src
        break

if found is not None:
    print(f"copying {NAME} ({found.stat().st_size / 1e6:.1f} MB) …", flush=True)
    shutil.copy2(found, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.1f} MB)", flush=True)
elif dst.is_file():
    print(f"already in clone: {dst} ({dst.stat().st_size / 1e6:.1f} MB)", flush=True)
else:
    print("not on Drive — next cell will build n=200 --tetm with xvfb-run", flush=True)

## HDF5 yoksa üret (`--tetm`, n=200)

MTGeophysics GLMakie import eder; sahte display için `xvfb-run` şart. Önceki hücre `OK` dediyse bu hücre hemen çıkar.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=0

H5=/content/PriorModel/data/synthetic/train_pairs_v8_tetm_n200.h5
if [[ -f "$H5" ]]; then
  echo "already have $H5 ($(du -h "$H5" | cut -f1)) — skip build"
  exit 0
fi

echo "=== install xvfb (GLMakie needs a fake display) ==="
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xvfb >/dev/null

mkdir -p /content/PriorModel/data/synthetic
echo "=== build n=200  --tetm  dx=80 m  (~30–60 min) ==="
script -q -c 'xvfb-run -a julia --project=/content/PriorModel \
  /content/PriorModel/scripts/build_train_pairs.jl \
  --n 200 --tetm \
  --out /content/PriorModel/data/synthetic/train_pairs_v8_tetm_n200.h5 \
  --seed 42' /dev/null

ls -lh "$H5"

## Eğit (50 epoch, TE+TM)

`--commemi-every 0` ve `--no-plot` Colab'da zorunlu (GLMakie).

İlk epoch CUDA/Zygote derler; 10–20 dk çıktısız donma normal. Sırayla bakın: `Device: CUDA GPU` → `epoch 1/50 starting` → `first GPU train step finished`.

In [ ]:
%%bash
set -euo pipefail
export GKSwstype=nul
export JULIA_PKG_PRECOMPILE_AUTO=1

H5=/content/PriorModel/data/synthetic/train_pairs_v8_tetm_n200.h5
test -f "$H5" || { echo "missing $H5 — run the xvfb build cell first"; exit 1; }
mkdir -p /content/PriorModel/models /content/PriorModel/results

echo "=== train TE+TM  50 ep  → models/prior_v8_tetm_n200.jld2 ==="
script -q -c "julia --project=/content/PriorModel \
  /content/PriorModel/src/training/train_mt_resistivity.jl \
  --dataset ${H5} \
  --epochs 50 \
  --output /content/PriorModel/models/prior_v8_tetm_n200.jld2 \
  --training-log /content/PriorModel/results/training_log_v8_tetm.csv \
  --split-json /content/PriorModel/results/train_val_split_v8_tetm.json \
  --curve-png /content/PriorModel/results/training_curve_v8_tetm.png \
  --commemi-every 0 --no-plot" /dev/null

## Checkpoint + HDF5'i Drive'a kopyala

Yerelde COMMEMI eval:

```bash
julia --project=. scripts/evaluate_mid_scale_v8.jl
```

Checkpoint'i `models/prior_v8_tetm_n200.jld2` olarak indirin. VFSA'yı Colab'da çalıştırmayın.

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/content/PriorModel")
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/PriorModelData/PriorModel/PriorModel"),
    Path("/content/drive/MyDrive/PriorModelData/PriorModel"),
    Path("/content/drive/MyDrive/PriorModel"),
]
DRIVE = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
DRIVE.mkdir(parents=True, exist_ok=True)

rels = (
    "models/prior_v8_tetm_n200.jld2",
    "data/synthetic/train_pairs_v8_tetm_n200.h5",
    "results/training_log_v8_tetm.csv",
    "results/train_val_split_v8_tetm.json",
)

print(f"=== copy TE+TM artifacts → {DRIVE} ===", flush=True)
for rel in rels:
    src = ROOT / rel
    dst = DRIVE / rel
    if not src.is_file():
        print(f"skip (missing): {src}", flush=True)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"OK {dst} ({dst.stat().st_size / 1e6:.2f} MB)", flush=True)